# Modeling

Treina e compara vários modelos de classificação usando `data_preparation.py` e `data_modeling.py`, e guarda o melhor em `models/`.

In [ ]:
import sys
from pathlib import Path

SRC_PATH = str(Path("../src").resolve())
sys.path.insert(0, SRC_PATH)  # garante prioridade ao m?dulo deste reposit?rio
sys.modules.pop("data_preparation", None)
sys.modules.pop("data_modelling", None)  # descarta eventual vers?o antiga j? carregada

import data_preparation as dp
import data_modelling as dm
from datetime import datetime

## 1. Preparar os dados

Reaproveita a fase anterior.

In [ ]:
DATASET_NAME = "malware"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_PATH = "../dataset/raw/Malware_and_benign_recognition.csv"

X_train, X_val, X_test, y_train, y_val, y_test = dp.run_data_preparation(DATA_PATH, target="Malicious")

## 2. Comparar modelos com validação cruzada

Testa Regressão Logística, kNN, SVM, Naive Bayes, MLP, Árvore de Decisão, Random Forest e Gradient Boosting, com `StratifiedKFold` (normalização aplicada só aos modelos que precisam).

In [ ]:
results_df = dm.evaluate_models(X_train, y_train, scoring="f1_macro")
dm.save_results_csv(results_df, dataset_name=DATASET_NAME, output_dir="../models", timestamp=RUN_TIMESTAMP)
results_df

In [ ]:
dm.plot_model_comparison(results_df, scoring="f1_macro", dataset_name=DATASET_NAME, output_dir="../models", timestamp=RUN_TIMESTAMP)

## 3. Escolher e treinar o modelo campeão

In [ ]:
best_name = dm.select_best_model(results_df)
tuned_pipe = dm.train_final_model(
    best_name, X_train, y_train, X_val, y_val, scoring="f1_macro",
)
validation_metrics = dm.evaluate_on_validation(
    tuned_pipe, X_train, y_train, X_val, y_val, scoring="f1_macro",
)

## 4. Avaliar no conjunto de teste

In [ ]:
best_pipe, X_train_val, y_train_val = dm.refit_final_model(
    tuned_pipe, X_train, y_train, X_val, y_val,
)
test_metrics = dm.evaluate_on_test(
    best_pipe, X_train_val, y_train_val, X_test, y_test,
    scoring="f1_macro", dataset_name=DATASET_NAME,
    output_dir="../models", timestamp=RUN_TIMESTAMP,
)

## 5. Guardar o modelo campe??o

O pipeline versionado e a c??pia `latest` ficam em `../models/`, acompanhados por metadados JSON com m??tricas e features. CSVs e gr??ficos de compara????o e avalia????o tamb??m s??o guardados nessa pasta com dataset e timestamp.

In [ ]:
dm.save_model(
    best_pipe, best_name, dataset_name=DATASET_NAME, output_dir="../models",
    metrics={"validation": validation_metrics, "test": test_metrics},
    feature_names=list(X_train.columns), timestamp=RUN_TIMESTAMP,
)


## Alternativa: correr tudo de uma vez

In [ ]:
# best_pipe = dm.run_modeling(X_train, X_val, X_test, y_train, y_val, y_test, dataset_name=DATASET_NAME)